#### **Base and Large model evaluation prior fine-tuning**

This notebook covers the evaluation of the project. It has two main purposes, both linked to the evaluation part of our project: 

* the assessment of flan-t5-base and flan-t5-large on the test split at four precision levels : **FP32, BF16, INT8, INT4**, using Exact Match (EM) and F1. 
* the build and explaination of the different functions that **evaluate.py** contain, ending with **evaluate_model()**, which is the function we'll use throughout the project to score a model. 

In order to run this notebook from top to bottom, you need to have cloned first the github repo.

**Let us first assess the two models on different quantisation levels**:

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.evaluate import evaluate_model
from src.preprocessing import load_processed
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch

ds = load_processed()
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

/opt/miniconda3/envs/deep_learning_project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
KWARGS = {
    "fp32": {},
    "bf16": {"torch_dtype": torch.bfloat16},
    "int8": {"load_in_8bit": True, "device_map": "auto"},
    "int4": {"load_in_4bit": True, "device_map": "auto"},
}

def load_model(model_name, precision_mode):

    if precision_mode not in KWARGS:
        raise ValueError(f"Unknown precision_mode: {precision_mode!r}")

    model = AutoModelForSeq2SeqLM.from_pretrained(
        f"google/{model_name}", **KWARGS[precision_mode]
    )

    if precision_mode in ("fp32", "bf16"):
        model = model.to(device)

    return model

In [5]:
import pandas as pd

configs = [
    ("flan-t5-base",  "fp32"),
    ("flan-t5-base",  "bf16"),
    ("flan-t5-base",  "int8"),
    ("flan-t5-base",  "int4"),
    ("flan-t5-large", "fp32"),
    ("flan-t5-large", "bf16"),
    ("flan-t5-large", "int8"),
    ("flan-t5-large", "int4"),
]

rows = []

for model_name, precision_mode in configs:
    print(f"--- {model_name} / {precision_mode} ---")

    model = load_model(model_name, precision_mode)
    row, preds = evaluate_model(
        model, tokenizer, ds["test"],
        model_name=model_name,
        state="raw",
        precision_mode=precision_mode,
    )
    rows.append(row)
    print(row)

    del model
    if device == 'mps': torch.mps.empty_cache() 
    else: torch.cuda.empty_cache()

pd.DataFrame(rows)

--- flan-t5-base / fp32 ---


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 21035.67it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
generating:  16%|█▌        | 60/375 [00:09<00:52,  6.06it/s]


KeyboardInterrupt: 

#### **evaluate.py construction**
Below is the experimentation and testing that led up to the construction of the evaluate.py file, with the building of all the different functions used when we call evaluate_model().



We first need a function to normalize the text that we'll use on the target answer and the predicted text for our models

In [1]:
import re
import string


def normalize_answer(s: str) -> str:

    def lower(text):
        return text.lower()

    def remove_punc(text):
        exclude = string.punctuation
        return "".join(ch for ch in text if ch not in exclude)

    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    return white_space_fix(remove_articles(remove_punc(lower(s))))

Let us now define the different function that we will use for evaluation.

First the Exact Match (EM) score:

In [2]:
def exact_match_score(prediction: str, target: str) -> int:
    #1 if the two strings match after normalization, 0 otherwise.
    return int(normalize_answer(prediction) == normalize_answer(target))

In [3]:
tests = [
    ("Paris", "Paris", 1),
    ("paris", "Paris", 1),
    ("The Norman conquest.", "Norman conquest", 1),
    ("Norman conquest of England","the Norman conquest", 0),
    ("1066", "in 1066", 0),
    ("", "Paris", 0),
    ("", "", 1),
]

for pred, target, expected in tests:
    got = exact_match_score(pred, target)
    flag = "ok" if got == expected else "MISMATCH"
    print(f"{pred!r:30} vs {target!r:22} -> {got}  (expected {expected})  {flag}")

'Paris'                        vs 'Paris'                -> 1  (expected 1)  ok
'paris'                        vs 'Paris'                -> 1  (expected 1)  ok
'The Norman conquest.'         vs 'Norman conquest'      -> 1  (expected 1)  ok
'Norman conquest of England'   vs 'the Norman conquest'  -> 0  (expected 0)  ok
'1066'                         vs 'in 1066'              -> 0  (expected 0)  ok
''                             vs 'Paris'                -> 0  (expected 0)  ok
''                             vs ''                     -> 1  (expected 1)  ok


Now the F1 score metric, which represent the harmonic mean of precision and recall

In [4]:
from collections import Counter

def f1_score(prediction: str, target: str) -> dict:

    pred_tokens = normalize_answer(prediction).split()
    target_tokens = normalize_answer(target).split()

    # Empty-string guard: full credit only if both sides are empty.
    if len(pred_tokens) == 0 or len(target_tokens) == 0:
        score = float(pred_tokens == target_tokens)
        return {"precision": score, "recall": score, "f1": score}

    common = Counter(pred_tokens) & Counter(target_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    precision = num_same / len(pred_tokens)
    recall = num_same / len(target_tokens)
    f1 = 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}

In [5]:
tests = [
    ("Paris",                      "Paris"),
    ("1066",                       "in 1066"),
    ("I forgot to say hello you",  "hello you"),
    ("Norman conquest of England", "the Norman conquest"),
    ("new york new york",          "new york new york"),
    ("new new new york",           "new york"),
    ("completely wrong",           "Paris"),
    ("",                           "Paris"),
    ("you hello",                  "hello you"),
]

for pred, gold in tests:
    s = f1_score(pred, gold)
    em = exact_match_score(pred, gold)
    print(f"{pred!r:30} vs {gold!r:22} "
          f"P={s['precision']:.3f} R={s['recall']:.3f} F1={s['f1']:.3f}  EM={em}")

'Paris'                        vs 'Paris'                P=1.000 R=1.000 F1=1.000  EM=1
'1066'                         vs 'in 1066'              P=1.000 R=0.500 F1=0.667  EM=0
'I forgot to say hello you'    vs 'hello you'            P=0.333 R=1.000 F1=0.500  EM=0
'Norman conquest of England'   vs 'the Norman conquest'  P=0.500 R=1.000 F1=0.667  EM=0
'new york new york'            vs 'new york new york'    P=1.000 R=1.000 F1=1.000  EM=1
'new new new york'             vs 'new york'             P=0.500 R=1.000 F1=0.667  EM=0
'completely wrong'             vs 'Paris'                P=0.000 R=0.000 F1=0.000  EM=0
''                             vs 'Paris'                P=0.000 R=0.000 F1=0.000  EM=0
'you hello'                    vs 'hello you'            P=1.000 R=1.000 F1=1.000  EM=0


Our functions so far work for one example of target output and predicted output. We now need to make it work for a whole batch.

In [ ]:
def compute_metrics(predictions: list, references: list) -> dict:
    
    #Average EM, precision, recall and F1 over a batch
    #predictions: list of strings of all decoded model output per example
    #references:  list of strings of all target answer per example
    
    if len(predictions) != len(references):
        raise ValueError(
            f"Length mismatch: {len(predictions)} predictions "
            f"vs {len(references)} references"
        )

    em_total = 0.0
    precision_total = 0.0
    recall_total = 0.0
    f1_total = 0.0

    for pred, target in zip(predictions, references):
        em_total += exact_match_score(pred, target)
        scores = f1_score(pred, target)
        precision_total += scores["precision"]
        recall_total += scores["recall"]
        f1_total += scores["f1"]

    n = len(predictions)

    return {
        "exact_match": round(em_total / n,5),
        "precision": round(precision_total / n,5),
        "recall": round(recall_total / n,5),
        "f1": round(f1_total / n,5),
        "n_examples": n,
    }

In [7]:
preds = [
    "Paris",
    "1066",
    "I forgot to say hello you",
    "completely wrong",
]
golds = [
    "Paris",
    "in 1066",
    "hello you",
    "Paris",
]

results = compute_metrics(preds, golds)
for k, v in results.items():
    print(f"{k:14} {v}")

exact_match    0.25
precision      0.58
recall         0.62
f1             0.54
n_examples     4


We now need to build the function that will generate the prediction with the model over a batch of examples

In [8]:
import sys
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import MAX_GEN_TOKENS, EVAL_BATCH_SIZE, DATA_PROCESSED, RESULTS_DIR

MODEL_NAME = "google/flan-t5-base"


device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device: {device}")

dataset = load_from_disk(DATA_PROCESSED)
print(dataset)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

device: mps
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 27000
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
})


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [9]:
test_set = dataset["test"]
sample = test_set.select(range(50))
print(f"full test set size: {len(test_set)} | sample size: {len(sample)}")

full test set size: 3000 | sample size: 50


In [10]:
from tqdm.auto import tqdm

def generate_predictions(model, tokenizer, dataset,
                         batch_size=EVAL_BATCH_SIZE,
                         max_new_tokens=MAX_GEN_TOKENS) -> list:

    # Run greedy generation on a dataset and return the decoded answer strings

    model.eval()
    device = next(model.parameters()).device
    predictions = []

    for start in tqdm(range(0, len(dataset), batch_size), desc="generating"):
        batch = dataset[start:start + batch_size]

        inputs = tokenizer.pad(
            {
                "input_ids": batch["input_ids"],
                "attention_mask": batch["attention_mask"],
            },
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
            )

        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        predictions.extend(decoded)

    return predictions

In [11]:
sample

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 50
})

In [12]:
preds = generate_predictions(model, tokenizer, sample, batch_size=10)

for i in range(len(sample)):
    print(f"Question: {sample[i]['question']}")
    print(f"target: {sample[i]['answers']['text'][0]}")
    print(f"pred: {preds[i]!r}\n")

generating:   0%|          | 0/5 [00:00<?, ?it/s]

Question: Where is the Hoppings funfair held?
target: Town Moor
pred: 'Newcastle'

Question: Which park in England has an alliterative name?
target: Hampstead Heath
pred: 'Town Moor'

Question: What is a soccer organization called in England?
target: Club
pred: 'Football Club'

Question: Where is the Town Moor?
target: Newcastle
pred: 'Newcastle'

Question: What makes Town Moor suitable to graze cattle on it?
target: green space
pred: "It is larger than London's famous Hyde Park and Hampstead Heath put together"

Question: Where do the owners of the cattle that graze in the Town Moor live?
target: Newcastle
pred: 'Newcastle'

Question: Which is possibly found in Town Moor ?
target: cattle
pred: 'cattle'

Question: Which actors are honorary freemen?
target: the Royal Shakespeare Company
pred: 'Bob Geldof, King Harald V of Norway, Bobby Robson, Alan Shearer, the late Nelson Mandela and the Royal Shakespeare Company'

Question: Who shares a name with an older type of transportation?
targe

In [13]:
targets = [example["answers"]["text"][0] for example in sample]
print(compute_metrics(preds, targets))

{'exact_match': 0.28, 'precision': 0.5, 'recall': 0.53, 'f1': 0.48, 'n_examples': 50}


Finally, we need now to build the function that merges computes_metrics, and generate_predictions

In [14]:
def evaluate_model(model, tokenizer, dataset,
                   model_name, state, precision_mode,
                   batch_size=EVAL_BATCH_SIZE,
                   max_new_tokens=MAX_GEN_TOKENS):
    
    predictions = generate_predictions(
        model, tokenizer, dataset,
        batch_size=batch_size,
        max_new_tokens=max_new_tokens,
    )

    references = [example["text"][0] for example in dataset["answers"]]
    metrics = compute_metrics(predictions, references)

    results_row = {
        "model_name": model_name,
        "state": state,
        "precision_mode": precision_mode,
        **metrics,
    }

    return results_row, predictions

In [15]:
row, preds = evaluate_model(
    model, tokenizer, sample,
    model_name="flan-t5-base",
    state="raw",
    precision_mode="fp32",
)

for k, v in row.items():
    print(f"{k:16} {v}")

generating:   0%|          | 0/7 [00:00<?, ?it/s]

model_name       flan-t5-base
state            raw
precision_mode   fp32
exact_match      0.28
precision        0.5
recall           0.53
f1               0.48
n_examples       50


Let us now perform the evaluation on the test dataset

In [16]:
row, preds = evaluate_model(
    model, tokenizer, test_set,
    model_name="flan-t5-base",
    state="raw",
    precision_mode="fp32",
)

for k, v in row.items():
    print(f"{k:16} {v}")

generating:   0%|          | 0/375 [00:00<?, ?it/s]

model_name       flan-t5-base
state            raw
precision_mode   fp32
exact_match      0.42
precision        0.56
recall           0.56
f1               0.53
n_examples       3000
